In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/HabitusAI')
sys.path.insert(0, '/content/drive/MyDrive/HabitusAI/Agents')
print("✅ Drive mounted")


In [8]:
!pip install transformers ultralytics segment-anything \
             torch torchvision Pillow numpy \
             opencv-python-headless pydantic -q

import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ CUDA: {torch.cuda.is_available()}")


AssertionError: Torch not compiled with CUDA enabled

In [7]:
import sys, os, json, urllib.request
from io import BytesIO
import numpy as np
from PIL import Image
import torch, cv2
from transformers import pipeline
from ultralytics import YOLO

# ── Config ────────────────────────────────────────────────
IMAGE_URL = "https://images.unsplash.com/photo-1618220179428-22790b461013?w=1080"
DEPTH_OUT  = "/tmp/depth.png"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load image ────────────────────────────────────────────
print("Loading image...")
with urllib.request.urlopen(IMAGE_URL) as r:
    pil_img = Image.open(BytesIO(r.read())).convert("RGB")
np_rgb = np.array(pil_img)
bgr    = cv2.cvtColor(np_rgb, cv2.COLOR_RGB2BGR)
H, W   = bgr.shape[:2]

# ── Step 1: Depth Anything V2 ─────────────────────────────
print("Running Depth Anything V2...")
depth_pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=0 if DEVICE == "cuda" else -1,
)
depth_result = depth_pipe(pil_img)
depth_pil    = depth_result["depth"]          # PIL grayscale image
depth_arr    = np.array(depth_pil, dtype=np.float32)

depth_pil.save(DEPTH_OUT)
print(f"✅ Depth map saved → {DEPTH_OUT}")

# ── Step 2: YOLOv8 object detection ──────────────────────
print("Running YOLOv8...")
INTERIOR = {"chair","couch","bed","dining table","tv","laptop",
            "microwave","oven","sink","refrigerator","clock",
            "vase","bottle","cup","book","potted plant"}

yolo    = YOLO("yolov8m.pt")
results = yolo(bgr, verbose=False)[0]
objects = []
for box in results.boxes:
    cls = results.names[int(box.cls)]
    if cls not in INTERIOR: continue
    conf = float(box.conf)
    if conf < 0.4: continue
    x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].tolist()]
    objects.append({"label": cls, "confidence": round(conf,3),
                    "bbox": [x1,y1,x2,y2]})

print(f"✅ Detected {len(objects)} objects: {[o['label'] for o in objects]}")

# ── Step 3: Room dimension estimate ──────────────────────
median_d = float(np.median(depth_arr))
fov      = np.deg2rad(60)
w_m      = round(2 * median_d * np.tan(fov/2), 1)
h_m      = round(w_m / (W/H), 1)
d_m      = round(median_d * 1.5, 1)
room_dims = {"w_m": w_m, "h_m": h_m, "d_m": d_m}
print(f"✅ Room dims (estimate): {room_dims}")

# ── Output ────────────────────────────────────────────────
output = {
    "depth_map_path": DEPTH_OUT,
    "objects": objects,
    "room_dims_estimate": room_dims,
}
print("\n=== VisionOutput ===")
print(json.dumps(output, indent=2))


✅ Ready


In [4]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(pil_img)
axes[0].set_title("Original Room")
axes[0].axis("off")
axes[1].imshow(depth_pil, cmap="inferno")
axes[1].set_title("Depth Map (Depth Anything V2)")
axes[1].axis("off")
plt.tight_layout()
plt.show()
